<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/13-evaluating-classifiers/02_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 13: Exercises

Twelve exercises: four Easy, five Medium and three Hard. You'll build the evaluation toolkit of this module from scratch: the confusion matrix and the metrics computed from it, ROC curves and AUC (two different ways), average precision, a calibration check, the cost-optimal threshold, Platt scaling, and a paired bootstrap for comparing two models.

Most checks compare your code with scikit-learn's own version of the same metric, so when they pass, you know your version computes exactly the same thing, including the awkward cases: ties, empty classes and divisions by zero.

**Run the setup cell first.**

Rules:

- NumPy, pandas and SciPy. Don't call scikit-learn's version of the metric you're writing; the checks use it to test your answer.
- Solutions are in `03_solutions.ipynb`.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import math
import warnings

import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix, f1_score, fbeta_score,
                             precision_score, recall_score, roc_auc_score, roc_curve)


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, (pd.DataFrame, pd.Series)) or isinstance(b, (pd.DataFrame, pd.Series)):
        if type(a) is not type(b):
            return False
        options = dict(check_dtype=False, check_exact=False, rtol=tol, atol=tol, check_index_type=False)
        try:
            if isinstance(a, pd.DataFrame):
                pd.testing.assert_frame_equal(a, b, check_column_type=False, **options)
            else:
                pd.testing.assert_series_equal(a, b, **options)
            return True
        except AssertionError:
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, (pd.DataFrame, pd.Series)):
        lines = (x.to_string(max_rows=6, max_cols=6) if isinstance(x, pd.DataFrame) else x.to_string(max_rows=6)).splitlines()
        return f"{type(x).__name__} shape={x.shape}\n          " + "\n          ".join(lines[:8])
    if isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, tuple) and any(isinstance(i, np.ndarray) for i in x):
        text = "(" + ", ".join(_short(i, limit) for i in x) + ")"
        return text
    else:
        text = repr(x)
    if isinstance(x, (pd.DataFrame, pd.Series)):
        return text
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def check_confusion_counts(f):
    rng = np.random.default_rng(0)
    y = rng.integers(0, 2, 200)
    p = rng.integers(0, 2, 200)
    tn_r, fp_r, fn_r, tp_r = confusion_matrix(y, p, labels=[0, 1]).ravel()
    _scenarios("confusion_counts", [
        ("the example", lambda: f([1, 0, 1, 1, 0], [1, 1, 0, 1, 0]), {"tp": 2, "fp": 1, "fn": 1, "tn": 1},
         "TP: predicted 1 and actually 1; FP: predicted 1 but actually 0; and so on"),
        ("matches scikit-learn on random labels", lambda: f(y, p),
         {"tp": int(tp_r), "fp": int(fp_r), "fn": int(fn_r), "tn": int(tn_r)}, None),
        ("bools work", lambda: f([True, False, True], [True, True, False]), {"tp": 1, "fp": 1, "fn": 1, "tn": 0}, None),
        ("no positives at all", lambda: f([0, 0, 0], [0, 0, 0]), {"tp": 0, "fp": 0, "fn": 0, "tn": 3}, None),
        ("values are Python ints", lambda: sorted({type(v).__name__ for v in f(np.array([1, 0]), np.array([1, 1])).values()}),
         ["int"], "wrap each count in int(...)"),
        ("different lengths raise", lambda: f([1, 0, 1], [1, 0]), Raises(ValueError), None),
        ("a single prediction for three labels raises", lambda: f([1, 0, 1], [1]), Raises(ValueError),
         "NumPy would broadcast one value against three, so compare the lengths yourself"),
        ("probabilities instead of labels raise", lambda: f([1, 0, 1], [0.9, 0.2, 0.6]), Raises(ValueError),
         "check that every label is 0 or 1"),
        ("a label of 2 raises", lambda: f([1, 2, 0], [1, 0, 0]), Raises(ValueError), None),
    ])


def check_binary_metrics(f):
    rng = np.random.default_rng(1)
    y = rng.integers(0, 2, 300)
    p = np.where(rng.random(300) < 0.7, y, 1 - y)
    ref = {
        "precision": precision_score(y, p), "recall": recall_score(y, p),
        "specificity": recall_score(y, p, pos_label=0), "f1": f1_score(y, p),
        "accuracy": accuracy_score(y, p),
        "balanced_accuracy": (recall_score(y, p) + recall_score(y, p, pos_label=0)) / 2,
    }
    _scenarios("binary_metrics", [
        ("a small example", lambda: f([1, 1, 1, 0, 0, 0, 0, 0], [1, 1, 0, 1, 0, 0, 0, 0]),
         {"precision": 2 / 3, "recall": 2 / 3, "specificity": 0.8, "f1": 2 / 3, "accuracy": 0.75,
          "balanced_accuracy": (2 / 3 + 0.8) / 2}, "TP=2, FP=1, FN=1, TN=4"),
        ("matches scikit-learn on random predictions", lambda: f(y, p), ref, None),
        ("nothing predicted positive: precision and F1 are 0.0", lambda: f([1, 0, 0, 1], [0, 0, 0, 0]),
         {"precision": 0.0, "recall": 0.0, "specificity": 1.0, "f1": 0.0, "accuracy": 0.5, "balanced_accuracy": 0.5},
         "when a denominator is 0, use 0.0"),
        ("no actual positives: recall is 0.0", lambda: f([0, 0, 0], [1, 0, 0])["recall"], 0.0, None),
        ("returns floats", lambda: sorted({type(v).__name__ for v in f([1, 0], [1, 0]).values()}), ["float"], None),
    ])


def check_f_beta(f):
    _scenarios("f_beta", [
        ("F1 of 0.5 and 1.0 is 2/3", lambda: f(0.5, 1.0), 2 / 3, "with beta = 1 this is 2PR / (P + R)"),
        ("F1 punishes imbalance: 1.0 and 0.1", lambda: f(1.0, 0.1), 2 * 0.1 / 1.1, None),
        ("F2 example", lambda: f(0.5, 1.0, beta=2), 5 * 0.5 / (4 * 0.5 + 1), None),
        ("recall-heavy model: F2 > F1 > F0.5", lambda: f(0.5, 1.0, 2) > f(0.5, 1.0) > f(0.5, 1.0, 0.5), True,
         "beta > 1 weights recall more"),
        ("matches scikit-learn's fbeta_score", lambda: f(precision_score([1, 1, 0, 0, 1], [1, 0, 1, 1, 1]),
                                                         recall_score([1, 1, 0, 0, 1], [1, 0, 1, 1, 1]), beta=0.5),
         fbeta_score([1, 1, 0, 0, 1], [1, 0, 1, 1, 1], beta=0.5), None),
        ("both zero gives 0.0", lambda: f(0.0, 0.0), 0.0, "avoid dividing by zero"),
        ("returns a float", lambda: type(f(1, 1)).__name__, "float", None),
        ("beta = 0 raises", lambda: f(0.5, 0.5, beta=0), Raises(ValueError), None),
        ("precision above 1 raises", lambda: f(1.5, 0.5), Raises(ValueError), None),
    ])


def check_multiclass_confusion(f):
    y = ["cat", "dog", "cat", "bird", "dog", "cat"]
    p = ["cat", "cat", "cat", "dog", "dog", "bird"]
    rng = np.random.default_rng(2)
    y_num = rng.integers(0, 4, 200)
    p_num = np.where(rng.random(200) < 0.6, y_num, rng.integers(0, 4, 200))
    _scenarios("multiclass_confusion", [
        ("counts match scikit-learn (actual in rows)", lambda: f(y, p).to_numpy().tolist(),
         confusion_matrix(y, p).tolist(), "rows are actual classes, columns predicted"),
        ("default labels are sorted", lambda: (list(f(y, p).index), list(f(y, p).columns)),
         (["bird", "cat", "dog"], ["bird", "cat", "dog"]), "sorted(set(y_true) | set(y_pred))"),
        ("axis names", lambda: (f(y, p).index.name, f(y, p).columns.name), ("actual", "predicted"), None),
        ("labels order is respected", lambda: f(y, p, labels=["dog", "cat", "bird"]).to_numpy().tolist(),
         confusion_matrix(y, p, labels=["dog", "cat", "bird"]).tolist(), None),
        ("a predicted class that never occurs gets a row and column", lambda: f([0, 0], [0, 1]).to_numpy().tolist(),
         [[1, 1], [0, 0]], "use classes from both y_true and y_pred"),
        ("a class never predicted still gets a column", lambda: f([0, 1, 2], [0, 0, 0])[2].tolist(), [0, 0, 0], None),
        ("an extra label gets zeros", lambda: f([0, 1], [0, 1], labels=[0, 1, 5]).to_numpy().tolist(),
         [[1, 0, 0], [0, 1, 0], [0, 0, 0]], None),
        ("random integers match scikit-learn", lambda: f(y_num, p_num).to_numpy().tolist(),
         confusion_matrix(y_num, p_num).tolist(), None),
        ("different lengths raise", lambda: f([1, 2], [1]), Raises(ValueError), None),
    ])


def check_averaged_f1(f):
    rng = np.random.default_rng(3)
    y = rng.choice(["A", "B", "C", "D"], 400, p=[0.5, 0.3, 0.15, 0.05])
    p = np.where(rng.random(400) < 0.65, y, rng.choice(["A", "B", "C", "D"], 400))
    # a model that never predicts the small class "small"
    y_neglect = np.array(["big"] * 90 + ["small"] * 10)
    p_neglect = np.array(["big"] * 100)

    def ref(yt, yp):
        return {a: f1_score(yt, yp, average=a, zero_division=0) for a in ["macro", "weighted", "micro"]}

    _scenarios("averaged_f1", [
        ("matches scikit-learn on four classes", lambda: f(y, p), ref(y, p),
         "per class F1 = 2TP / (2TP + FP + FN); weights for 'weighted' are the counts in y_true"),
        ("micro F1 equals accuracy", lambda: f(y, p)["micro"], float(np.mean(y == p)),
         "pool TP, FP and FN over all classes first"),
        ("a neglected class drags macro down", lambda: f(y_neglect, p_neglect), ref(y_neglect, p_neglect),
         "a class with no correct predictions has F1 0.0"),
        ("a predicted class that never occurs counts in macro", lambda: f(["a", "a", "b"], ["a", "c", "b"]),
         ref(["a", "a", "b"], ["a", "c", "b"]), "use every class in y_true or y_pred"),
        ("perfect predictions", lambda: f([1, 2, 3, 3], [1, 2, 3, 3]), {"macro": 1.0, "weighted": 1.0, "micro": 1.0}, None),
    ])


def check_roc_curve_points(f):
    rng = np.random.default_rng(4)
    y = rng.integers(0, 2, 300)
    s = rng.normal(size=300) + y
    s_tied = np.round(s, 1)

    def ref(yt, sc):
        fpr, tpr, thr = roc_curve(yt, sc, drop_intermediate=False)
        return fpr, tpr, thr

    def as_tuple(r):
        return tuple(np.asarray(v, dtype=float) for v in r)

    _scenarios("roc_curve_points", [
        ("a small example with no ties", lambda: as_tuple(f([0, 0, 1, 1], [0.1, 0.4, 0.35, 0.8])),
         (np.array([0, 0, 0.5, 0.5, 1]), np.array([0, 0.5, 0.5, 1, 1]), np.array([np.inf, 0.8, 0.4, 0.35, 0.1])),
         "sort scores from high to low; positives move up, negatives move right"),
        ("ties make one step", lambda: as_tuple(f([0, 1, 1, 0], [0.5, 0.5, 0.9, 0.1])),
         (np.array([0, 0, 0.5, 1]), np.array([0, 0.5, 1, 1]), np.array([np.inf, 0.9, 0.5, 0.1])),
         "keep only one point for each distinct score"),
        ("matches scikit-learn on random scores", lambda: as_tuple(f(y, s)), ref(y, s), None),
        ("matches scikit-learn with many ties", lambda: as_tuple(f(y, s_tied)), ref(y, s_tied), None),
        ("works with lists", lambda: len(f([0, 1], [0.2, 0.7])[0]), 3, None),
        ("one class only raises", lambda: f([1, 1, 1], [0.2, 0.5, 0.9]), Raises(ValueError), None),
    ])


def check_auc_by_ranks(f):
    rng = np.random.default_rng(5)
    y = rng.integers(0, 2, 2000)
    s = rng.normal(size=2000) + 0.8 * y
    s_tied = np.round(s)
    y_big = (rng.random(200_000) < 0.1).astype(int)
    s_big = rng.normal(size=200_000) + y_big
    _scenarios("auc_by_ranks", [
        ("a perfect ranking", lambda: f([0, 0, 1, 1], [0.1, 0.2, 0.8, 0.9]), 1.0, None),
        ("a reversed ranking", lambda: f([1, 1, 0, 0], [0.1, 0.2, 0.8, 0.9]), 0.0, None),
        ("a small example: 3 of 4 pairs won", lambda: f([0, 0, 1, 1], [0.1, 0.4, 0.35, 0.8]), 0.75, None),
        ("ties count half", lambda: f([0, 1], [0.5, 0.5]), 0.5, "use average ranks for ties: rankdata does this"),
        ("all scores equal gives 0.5", lambda: f(y, np.zeros(2000)), 0.5, None),
        ("matches roc_auc_score", lambda: f(y, s), roc_auc_score(y, s), "AUC = (R - n_pos (n_pos + 1) / 2) / (n_pos n_neg)"),
        ("matches roc_auc_score with ties", lambda: f(y, s_tied), roc_auc_score(y, s_tied), None),
        ("200,000 examples", lambda: f(y_big, s_big), roc_auc_score(y_big, s_big), "don't compare every pair"),
        ("returns a float", lambda: type(f([0, 1], [0.1, 0.2])).__name__, "float", None),
        ("one class only raises", lambda: f([0, 0], [0.1, 0.2]), Raises(ValueError), None),
    ])


def check_average_precision(f):
    rng = np.random.default_rng(6)
    y = (rng.random(3000) < 0.1).astype(int)
    s = rng.normal(size=3000) + 1.5 * y
    s_tied = np.round(s, 1)
    y_more_neg = np.concatenate([y, np.zeros(9000, dtype=int)])
    s_more_neg = np.concatenate([s, rng.normal(size=9000)])
    _scenarios("average_precision", [
        ("a small example", lambda: f([1, 0, 1, 0], [0.9, 0.8, 0.7, 0.1]), 0.5 * 1 + 0.5 * 2 / 3,
         "recall rises by 0.5 at precision 1, then by 0.5 at precision 2/3"),
        ("a perfect ranking gives 1", lambda: f([0, 1, 0, 1], [0.1, 0.9, 0.2, 0.8]), 1.0, None),
        ("matches scikit-learn", lambda: f(y, s), average_precision_score(y, s), None),
        ("matches scikit-learn with ties", lambda: f(y, s_tied), average_precision_score(y, s_tied),
         "handle tied scores together"),
        ("all scores tied gives the base rate", lambda: f(y, np.ones(3000)), float(y.mean()), None),
        ("more negatives lower the AP", lambda: f(y_more_neg, s_more_neg) < f(y, s), True, None),
        ("returns a float", lambda: type(f([0, 1], [0.1, 0.2])).__name__, "float", None),
        ("no positives raises", lambda: f([0, 0, 0], [0.1, 0.5, 0.2]), Raises(ValueError), None),
    ])


def check_reliability_table(f):
    rng = np.random.default_rng(7)
    p = rng.random(5000)
    y_cal = (rng.random(5000) < p).astype(int)                       # calibrated by construction
    p_over = np.clip(0.5 + 1.6 * (p - 0.5), 0, 1)                     # the same outcomes, overconfident probabilities
    frac_ref, mean_ref = calibration_curve(y_cal, p, n_bins=10, strategy="uniform")

    def ece_ref(yt, pr, n_bins=10):
        bins = np.minimum((pr * n_bins).astype(int), n_bins - 1)
        return float(sum(np.mean(bins == b) * abs(yt[bins == b].mean() - pr[bins == b].mean()) for b in np.unique(bins)))

    _scenarios("reliability_table", [
        ("a hand-made example", lambda: f([0, 1, 1, 0], [0.1, 0.2, 0.9, 0.7], n_bins=2)[0],
         pd.DataFrame({"mean_predicted": [0.15, 0.8], "fraction_positive": [0.5, 0.5], "count": [2, 2]}),
         "columns mean_predicted, fraction_positive, count; one row per non-empty bin"),
        ("its ECE", lambda: f([0, 1, 1, 0], [0.1, 0.2, 0.9, 0.7], n_bins=2)[1], 0.5 * 0.35 + 0.5 * 0.3, None),
        ("matches calibration_curve", lambda: (f(y_cal, p)[0]["fraction_positive"].to_numpy(), f(y_cal, p)[0]["mean_predicted"].to_numpy()),
         (frac_ref, mean_ref), None),
        ("bin edges: 0.5 and 1.0 both go in the last of 2 bins", lambda: f([0, 1, 1], [0.0, 0.5, 1.0], n_bins=2)[0]["count"].tolist(),
         [1, 2], "np.minimum((p * n_bins).astype(int), n_bins - 1)"),
        ("empty bins are left out", lambda: len(f([0, 1], [0.05, 0.95])[0]), 2, None),
        ("counts are ints", lambda: f(y_cal, p)[0]["count"].dtype.kind, "i", None),
        ("calibrated model: small ECE", lambda: f(y_cal, p)[1], ece_ref(y_cal, p), None),
        ("overconfident model: larger ECE", lambda: f(y_cal, p_over)[1] > 3 * f(y_cal, p)[1], True, None),
        ("probability above 1 raises", lambda: f([0, 1], [0.5, 1.2]), Raises(ValueError), None),
    ])


def check_best_threshold(f):
    rng = np.random.default_rng(8)
    y = rng.integers(0, 2, 400)
    s = np.round(rng.normal(size=400) + y, 1)

    def brute(yt, sc, c_fp, c_fn):
        best = (np.inf, c_fn * yt.sum())
        for t in np.sort(np.unique(sc))[::-1]:
            pred = sc >= t
            cost = c_fp * np.sum(pred & (yt == 0)) + c_fn * np.sum(~pred & (yt == 1))
            if cost < best[1]:
                best = (float(t), float(cost))
        return float(best[0]), float(best[1])

    p_big = rng.random(20_000)
    y_big = (rng.random(20_000) < p_big).astype(int)

    _scenarios("best_threshold", [
        ("a small example", lambda: f([0, 1, 1, 0, 1], [0.1, 0.4, 0.8, 0.6, 0.3], 1, 1), (0.3, 1.0),
         "try every distinct score as a threshold, plus inf"),
        ("costly false alarms raise the threshold", lambda: f([0, 1, 1, 0, 1], [0.1, 0.4, 0.8, 0.6, 0.3], 3, 1), (0.8, 2.0), None),
        ("costly false alarms: flag nothing", lambda: f([0, 1, 0, 0], [0.9, 0.5, 0.4, 0.1], 100, 1), (np.inf, 1.0),
         "np.inf (predict nothing positive) is a candidate"),
        ("ties in cost: the highest threshold wins", lambda: f([1, 0, 1], [0.9, 0.5, 0.2], 1, 1), (0.9, 1.0), None),
        ("tied scores handled together, matches brute force", lambda: f(y, s, 1, 3), brute(y, s, 1, 3), None),
        ("matches brute force with other costs", lambda: f(y, s, 5, 2), brute(y, s, 5, 2), None),
        ("calibrated probabilities: close to cost_fp / (cost_fp + cost_fn)",
         lambda: abs(f(y_big, p_big, 1, 4)[0] - 0.2) < 0.03, True, None),
        ("returns floats", lambda: [type(v).__name__ for v in f([0, 1], [0.2, 0.7], 1, 1)], ["float", "float"], None),
    ])


def check_platt_scaling(f):
    rng = np.random.default_rng(9)
    s = rng.normal(size=4000) * 3
    y = (rng.random(4000) < 1 / (1 + np.exp(-(0.7 * s - 0.5)))).astype(int)
    ref = LogisticRegression(C=1e12, tol=1e-12, max_iter=10_000).fit(s[:, None], y)
    s2 = rng.normal(size=500)
    y2 = (rng.random(500) < 1 / (1 + np.exp(-(2.0 * s2 + 1.0)))).astype(int)
    ref2 = LogisticRegression(C=1e12, tol=1e-12, max_iter=10_000).fit(s2[:, None], y2)

    p_cal = rng.uniform(0.02, 0.98, 50_000)
    y_cal = (rng.random(50_000) < p_cal).astype(int)

    def rounded(r, digits=4):
        return tuple(round(float(v), digits) for v in r)

    _scenarios("platt_scaling", [
        ("matches unregularised logistic regression", lambda: rounded(f(s, y)),
         rounded((ref.coef_[0, 0], ref.intercept_[0])), "Newton's method: theta -= solve(H, g)"),
        ("a second dataset", lambda: rounded(f(s2, y2)), rounded((ref2.coef_[0, 0], ref2.intercept_[0])), None),
        ("close to the true values (0.7, -0.5) for lots of data",
         lambda: tuple(bool(abs(v - t) < 0.1) for v, t in zip(f(s, y), (0.7, -0.5))), (True, True), None),
        ("log-odds of calibrated probabilities give a = 1, b = 0",
         lambda: rounded(f(np.log(p_cal / (1 - p_cal)), y_cal), 1), (1.0, 0.0), None),
        ("returns floats", lambda: [type(v).__name__ for v in f(s2, y2)], ["float", "float"], None),
        ("one class only raises", lambda: f([0.1, 0.5, 0.9], [1, 1, 1]), Raises(ValueError), None),
    ])


def check_paired_bootstrap(f):
    rng = np.random.default_rng(10)
    y = (rng.random(300) < 0.3).astype(int)
    a = rng.normal(size=300) + 1.6 * y
    b = a + rng.normal(size=300) * 0.8
    y_rare = np.array([1] * 3 + [0] * 47)
    a_rare = rng.normal(size=50) + y_rare
    b_rare = rng.normal(size=50) + y_rare

    def ref(yt, sa, sb, metric, n_resamples=1000, seed=0):
        g = np.random.default_rng(seed)
        diffs = []
        for _ in range(n_resamples):
            idx = g.integers(0, len(yt), len(yt))
            if yt[idx].min() == yt[idx].max():
                continue
            diffs.append(metric(yt[idx], sa[idx]) - metric(yt[idx], sb[idx]))
        diffs = np.array(diffs)
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        return {"difference": float(metric(yt, sa) - metric(yt, sb)), "low": float(lo), "high": float(hi),
                "share_a_better": float(np.mean(diffs > 0))}

    def separate_intervals_overlap():
        g = np.random.default_rng(0)
        auc_a, auc_b = [], []
        for _ in range(1000):
            idx = g.integers(0, 300, 300)
            auc_a.append(roc_auc_score(y[idx], a[idx]))
            auc_b.append(roc_auc_score(y[idx], b[idx]))
        return bool(np.percentile(auc_a, 2.5) < np.percentile(auc_b, 97.5))

    _scenarios("paired_bootstrap", [
        ("matches the reference procedure", lambda: f(y, a, b, roc_auc_score), ref(y, a, b, roc_auc_score),
         "draw idx = rng.integers(0, n, n) for every resample, from one rng"),
        ("the seed and number of resamples are used", lambda: f(y, a, b, roc_auc_score, n_resamples=300, seed=5),
         ref(y, a, b, roc_auc_score, 300, 5), None),
        ("works with any metric", lambda: f(y, a, b, average_precision_score, n_resamples=200),
         ref(y, a, b, average_precision_score, 200), None),
        ("single-class resamples are skipped", lambda: f(y_rare, a_rare, b_rare, roc_auc_score, n_resamples=200),
         ref(y_rare, a_rare, b_rare, roc_auc_score, 200), "skip the resample if y_true[idx] has only one class"),
        ("separate intervals overlap, but the paired one excludes 0",
         lambda: (separate_intervals_overlap(), f(y, a, b, roc_auc_score)["low"] > 0), (True, True), None),
        ("different lengths raise", lambda: f([0, 1, 1], [0.1, 0.2], [0.3, 0.4, 0.5], roc_auc_score), Raises(ValueError), None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `confusion_counts`

Write `confusion_counts(y_true, y_pred)` that returns a dict with the four counts of a binary confusion matrix: `{"tp": ..., "fp": ..., "fn": ..., "tn": ...}`, as plain Python `int`s.

- Labels are 0 (negative) and 1 (positive). Accept lists or arrays, of ints or bools.
- Raise `ValueError` if the inputs have different lengths, or if any label is not 0 or 1.

Example: `confusion_counts([1, 0, 1, 1, 0], [1, 1, 0, 1, 0])` returns `{"tp": 2, "fp": 1, "fn": 1, "tn": 1}`.

In [2]:
def confusion_counts(y_true, y_pred) -> dict:
    # your code here
    pass

In [3]:
check_confusion_counts(confusion_counts)

FAIL  confusion_counts: the example
      expected: {'tp': 2, 'fp': 1, 'fn': 1, 'tn': 1}
      got:      None
      hint: TP: predicted 1 and actually 1; FP: predicted 1 but actually 0; and so on


### Exercise 2: `binary_metrics`

Write `binary_metrics(y_true, y_pred)` that returns a dict of six floats:

- `"precision"`: TP / (TP + FP)
- `"recall"`: TP / (TP + FN)
- `"specificity"`: TN / (TN + FP)
- `"f1"`: the harmonic mean of precision and recall
- `"accuracy"`: (TP + TN) / total
- `"balanced_accuracy"`: the average of recall and specificity

When a ratio is undefined because its denominator is 0 (for example, precision when nothing is predicted positive), use `0.0`. F1 is `0.0` when precision and recall are both 0.

Your `confusion_counts` from Exercise 1 is available, so you can build on it.

In [4]:
def binary_metrics(y_true, y_pred) -> dict:
    # your code here
    pass

In [5]:
check_binary_metrics(binary_metrics)

FAIL  binary_metrics: a small example
      expected: {'precision': 0.6666666666666666, 'recall': 0.6666666666666666, 'specificity': 0.8, 'f1...
      got:      None
      hint: TP=2, FP=1, FN=1, TN=4


### Exercise 3: `f_beta`

Write `f_beta(precision, recall, beta=1.0)` that returns the F-beta score as a float:

$$F_\beta = \frac{(1 + \beta^2) \cdot P \cdot R}{\beta^2 \cdot P + R}$$

- Return `0.0` when precision and recall are both 0.
- Raise `ValueError` if `beta` is not greater than 0, or if precision or recall is outside the range 0 to 1.

Example: `f_beta(0.5, 1.0)` is `2/3`, and `f_beta(1.0, 0.1)` is about `0.18`.

In [6]:
def f_beta(precision: float, recall: float, beta: float = 1.0) -> float:
    # your code here
    pass

In [7]:
check_f_beta(f_beta)

FAIL  f_beta: F1 of 0.5 and 1.0 is 2/3
      expected: 0.6666666666666666
      got:      None
      hint: with beta = 1 this is 2PR / (P + R)


### Exercise 4: `multiclass_confusion`

Write `multiclass_confusion(y_true, y_pred, labels=None)` that returns the confusion matrix as a pandas DataFrame:

- Rows are the **actual** classes, columns the **predicted** classes, both in the order of `labels`.
- If `labels` is `None`, use the sorted list of every class that appears in either `y_true` or `y_pred`.
- Values are integer counts. Name the index `"actual"` and the columns `"predicted"` (set `df.index.name` and `df.columns.name`).
- Raise `ValueError` if the inputs have different lengths.

Labels can be any values that can be sorted: ints, strings and so on.

In [8]:
def multiclass_confusion(y_true, y_pred, labels=None) -> pd.DataFrame:
    # your code here
    pass

In [9]:
check_multiclass_confusion(multiclass_confusion)

FAIL  multiclass_confusion: counts match scikit-learn (actual in rows)
      raised AttributeError: 'NoneType' object has no attribute 'to_numpy'
      expected: [[0, 0, 1], [1, 2, 0], [0, 1, 1]]
      hint: rows are actual classes, columns predicted


## Medium

### Exercise 5: `averaged_f1`

Write `averaged_f1(y_true, y_pred)` for a multiclass problem. It returns a dict with three floats: `"macro"`, `"weighted"` and `"micro"`.

For each class (every class that appears in `y_true` or `y_pred`), treat that class as positive and all others as negative, and compute its F1. A class's F1 is `0.0` when it's undefined. Then:

- `"macro"`: the plain average of the per-class F1 scores
- `"weighted"`: the average weighted by each class's **support** (how many times it appears in `y_true`)
- `"micro"`: add up TP, FP and FN over all classes first, then compute one F1 from the totals

Don't use scikit-learn's metric functions. `multiclass_confusion` from Exercise 4 is available if you want it.

In [10]:
def averaged_f1(y_true, y_pred) -> dict:
    # your code here
    pass

In [11]:
check_averaged_f1(averaged_f1)

FAIL  averaged_f1: matches scikit-learn on four classes
      expected: {'macro': 0.7037379201903107, 'weighted': 0.7960372382479655, 'micro': 0.7825}
      got:      None
      hint: per class F1 = 2TP / (2TP + FP + FN); weights for 'weighted' are the counts in y_true


### Exercise 6: `roc_curve_points`

Write `roc_curve_points(y_true, scores)` that returns three float arrays `(fpr, tpr, thresholds)`, one entry per point of the ROC curve.

- The first point is `(0, 0)` with threshold `np.inf` (nothing is predicted positive).
- Then one point for each **distinct** score, from highest to lowest. At threshold `t`, everything with `score >= t` is predicted positive.
- Tied scores must be handled together: they all cross the threshold at once, so they make a single (possibly diagonal) step.
- Raise `ValueError` if `y_true` doesn't contain both classes.

This matches `sklearn.metrics.roc_curve(y_true, scores, drop_intermediate=False)`, which the check compares with. Don't call it yourself.

In [12]:
def roc_curve_points(y_true, scores):
    # your code here
    pass

In [13]:
check_roc_curve_points(roc_curve_points)

FAIL  roc_curve_points: a small example with no ties
      raised TypeError: 'NoneType' object is not iterable
      expected: (array(shape=(5,), dtype=float64, [0. , 0. , 0.5, 0.5, 1. ]), array(shape=(5,), dtype=float64, [0. , 0.5, 0.5, 1. , 1. ]), array(shape=(5,), dtype=float64, [ inf, 0.8 , 0.4 , 0.35, 0.1 ]))
      hint: sort scores from high to low; positives move up, negatives move right


### Exercise 7: `auc_by_ranks`

Write `auc_by_ranks(y_true, scores)` that returns the ROC AUC as a float, using its meaning: the probability that a random positive scores higher than a random negative, with ties counting half.

Don't compare every pair (that's slow for large data). Use **ranks** instead:

1. Rank all scores from 1 (lowest) to n, giving tied scores the average of their ranks. `scipy.stats.rankdata` does this.
2. Add up the ranks of the positives: `R`.
3. `AUC = (R - n_pos (n_pos + 1) / 2) / (n_pos · n_neg)`

Raise `ValueError` if `y_true` doesn't contain both classes.

In [14]:
from scipy.stats import rankdata


def auc_by_ranks(y_true, scores) -> float:
    # your code here
    pass

In [15]:
check_auc_by_ranks(auc_by_ranks)

FAIL  auc_by_ranks: a perfect ranking
      expected: 1.0
      got:      None


### Exercise 8: `average_precision`

Write `average_precision(y_true, scores)` that returns the average precision (AP) as a float.

AP summarises the precision-recall curve as a weighted average of the precision at each threshold, weighted by how much recall increases there:

$$AP = \sum_k (R_k - R_{k-1}) \, P_k$$

Use one threshold for each **distinct** score, from highest to lowest (predict positive if `score >= threshold`), with `R_0 = 0`. Tied scores are handled together, as in Exercise 6.

- Raise `ValueError` if `y_true` contains no positives.

The check compares with `sklearn.metrics.average_precision_score`. Don't call it yourself.

In [16]:
def average_precision(y_true, scores) -> float:
    # your code here
    pass

In [17]:
check_average_precision(average_precision)

FAIL  average_precision: a small example
      expected: 0.8333333333333333
      got:      None
      hint: recall rises by 0.5 at precision 1, then by 0.5 at precision 2/3


### Exercise 9: `reliability_table`

Write `reliability_table(y_true, probs, n_bins=10)` for checking calibration. It returns `(table, ece)`:

- `table`: a DataFrame with one row per **non-empty** bin, in order, and columns `"mean_predicted"` (average probability in the bin), `"fraction_positive"` (fraction of positives in the bin) and `"count"` (number of examples, an int).
- `ece`: the **expected calibration error**, the weighted average gap between those two columns: `Σ (count / n) · |fraction_positive - mean_predicted|`, as a float.

Use equal-width bins: bin `i` holds probabilities `p` with `i / n_bins <= p < (i + 1) / n_bins`, and `p = 1.0` goes in the last bin. One way: `np.minimum((p * n_bins).astype(int), n_bins - 1)`.

Raise `ValueError` if any probability is outside 0 to 1.

In [18]:
def reliability_table(y_true, probs, n_bins: int = 10):
    # your code here
    pass

In [19]:
check_reliability_table(reliability_table)

FAIL  reliability_table: a hand-made example
      raised TypeError: 'NoneType' object is not subscriptable
      expected: DataFrame shape=(2, 3)
             mean_predicted  fraction_positive  count
          0            0.15                0.5      2
          1            0.80                0.5      2
      hint: columns mean_predicted, fraction_positive, count; one row per non-empty bin


## Hard

### Exercise 10: `best_threshold`

Write `best_threshold(y_true, scores, cost_fp, cost_fn)` that finds the threshold with the lowest total cost, and returns `(threshold, total_cost)` as two floats.

- Predict positive when `score >= threshold`. Total cost = `cost_fp × FP + cost_fn × FN`.
- The candidate thresholds are every distinct score, plus `np.inf` (predict nothing positive).
- If several thresholds tie for the lowest cost, return the **highest** of them.
- Make it fast: work it out for every candidate in one pass over sorted scores, with cumulative sums, rather than recounting the confusion matrix for each threshold. The check uses 20,000 examples.

In [20]:
def best_threshold(y_true, scores, cost_fp: float, cost_fn: float):
    # your code here
    pass

In [21]:
check_best_threshold(best_threshold)

FAIL  best_threshold: a small example
      expected: (0.3, 1.0)
      got:      None
      hint: try every distinct score as a threshold, plus inf


### Exercise 11: `platt_scaling`

Write `platt_scaling(scores, y)` that fits **Platt scaling**, the "sigmoid" calibration from the lesson. It finds `a` and `b` so that

$$p = \frac{1}{1 + e^{-(a \cdot s + b)}}$$

turns a model's raw scores `s` into calibrated probabilities. Return `(a, b)` as two floats.

- Choose `a` and `b` to minimise the **mean log loss**, using **Newton's method**: starting from `a = b = 0`, repeat `θ ← θ - H⁻¹ g` until the step is smaller than `1e-10` (at most 100 steps), where `g` is the gradient and `H` the Hessian (matrix of second derivatives) of the loss.
- For the log loss with `z = a·s + b` and `p = sigmoid(z)`: with `X = [s, 1]` as an `(n, 2)` matrix, `g = Xᵀ(p - y) / n` and `H = Xᵀ diag(p(1 - p)) X / n`.
- Raise `ValueError` if `y` doesn't contain both classes. You can assume the classes overlap (so the best `a` and `b` are finite).

The check compares with scikit-learn's unregularised `LogisticRegression` on one feature.

In [22]:
def platt_scaling(scores, y):
    # your code here
    pass

In [23]:
check_platt_scaling(platt_scaling)

FAIL  platt_scaling: matches unregularised logistic regression
      raised TypeError: 'NoneType' object is not iterable
      expected: (0.6869, -0.4385)
      hint: Newton's method: theta -= solve(H, g)


### Exercise 12: `paired_bootstrap`

Write `paired_bootstrap(y_true, scores_a, scores_b, metric, n_resamples=1000, seed=0)` that compares two models on the **same** test cases. It returns a dict:

- `"difference"`: `metric(y_true, scores_a) - metric(y_true, scores_b)` on the full data
- `"low"`, `"high"`: the 2.5th and 97.5th percentiles (`np.percentile`) of the bootstrap differences
- `"share_a_better"`: the fraction of bootstrap differences that are greater than 0

All four as floats. `metric` is a function like `roc_auc_score`, called as `metric(y, scores)`.

To make results reproducible, follow this procedure exactly:

- Create `rng = np.random.default_rng(seed)`, then repeat `n_resamples` times: draw `idx = rng.integers(0, n, n)`; if `y_true[idx]` contains only one class, skip it (don't redraw); otherwise compute the difference of the metric for the two models on `idx`.
- Raise `ValueError` if the three inputs don't have the same length.

In [24]:
def paired_bootstrap(y_true, scores_a, scores_b, metric, n_resamples: int = 1000, seed: int = 0) -> dict:
    # your code here
    pass

In [25]:
check_paired_bootstrap(paired_bootstrap)

FAIL  paired_bootstrap: matches the reference procedure
      expected: {'difference': 0.03658026755852839, 'low': 0.00403580433880941, 'high': 0.0677277372601...
      got:      None
      hint: draw idx = rng.integers(0, n, n) for every resample, from one rng


## Done?

Once everything passes, compare with `03_solutions.ipynb`. Then open the mini project in `04_mini_project.ipynb`, where these tools meet a real decision: which customers a Portuguese bank should call in a marketing campaign, judged by accuracy, ranking, calibration and profit, and tested on the future.